In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on 2024-05-27

@author: Juan Enrique López

@description: Jupyter Notebook creado para obtener las vulnerabilidades publicadas en NIST en los últimos n días.

"""

'\nCreated on 2024-05-27\n\n@author: Juan Enrique López\n\n@description: Jupyter Notebook creado para obtener las vulnerabilidades publicadas en NIST en los últimos n días.\n\n'

#### **NIST**

**Requerimientos**

In [11]:
import nvdlib
import datetime
import os
import requests.exceptions
from retrying import retry

**Parámetros**

In [12]:
save_as_md = True
end_date = datetime.datetime.now()
seach_days = 1 # Ha de ser un entero
start_date = end_date - datetime.timedelta(days=seach_days)
print(f"Se van a buscar vulnerabilidades entre las fechas {start_date.strftime('%d/%m/%Y')} y {end_date.strftime('%d/%m/%Y')}")

Se van a buscar vulnerabilidades entre las fechas 01/07/2024 y 02/07/2024


**Funciones**

In [13]:
@retry(wait_fixed=2000, stop_max_attempt_number=5, retry_on_exception=lambda e: isinstance(e, requests.exceptions.ReadTimeout))

def get_nist_vulnerabilities(s_date, e_date):
    '''
    Mediante esta función se llama a la librería nvdlib con el objetivo de retornar un objeto cve (pseudo diccionario) que contendrá las vulnerabilidades publicadas en NIST entre las fechas definidas. Se ha añadido un decorador (@retry) que se encargará de re-ejecutar la llamada un total de 5 ocasiones esperando un total de 2 segundos entre cada intento.
    '''
    results = nvdlib.searchCVE(pubStartDate=s_date, pubEndDate=e_date)
    return results

In [14]:
# Función encargada de construir las propiedades del fichero .md
def create_header_properties(cve):
    '''
    Con el objetivo de integrar en el fichero markdown final las propiedades por las que poder filtrar en obsidian. Mediante la función se mapean una serie de características (no siempre disponibles) y se retorna una cadena para poder añadir al principio del documento .md.
    '''
    cvss31_baseScore = cvss31_baseSeverity = cvss31_vectorString = cvss31_version = cvss31_impactScore = cvss31_exploitabilityScore = cvss31_attackVector = cvss31_attackComplexity = cvss31_privilegesRequired = cvss31_userInteraction = cvss31_scope = cvss31_confidentialityImpact = cvss31_integrityImpact = cvss31_availabilityImpact = cvss31_source = ''

    cna_baseScore = cna_baseSeverity = cna_vectorString = cna_version = cna_impactScore = cna_exploitabilityScore = cna_attackVector = cna_attackComplexity = cna_privilegesRequired = cna_userInteraction = cna_scope = cna_confidentialityImpact = cna_integrityImpact = cna_availabilityImpact = cna_source = ''

    cvss20_baseScore = cvss20_baseSeverity = cvss20_vectorString = cvss20_impactSubscore = cvss20_exploitabilitySubscore = cvss20_accessVector = cvss20_accessComplexity = cvss20_authentication = cvss20_confidentialityImpact = cvss20_integrityImpact = cvss20_availabilityImpact = ''

    if hasattr(cve, 'metrics'):
        if len(cve.metrics)>=1:
            if hasattr(cve.metrics, 'cvssMetricV31'):
                # Mapeamos las variavles 
                if cve.metrics.cvssMetricV31[0].source == 'nvd@nist.gov':
                    cvss31_baseScore =  cve.metrics.cvssMetricV31[0].cvssData.baseScore
                    cvss31_baseSeverity = cve.metrics.cvssMetricV31[0].cvssData.baseSeverity
                    cvss31_vectorString = cve.metrics.cvssMetricV31[0].cvssData.vectorString
                    cvss31_version = cve.metrics.cvssMetricV31[0].cvssData.version
                    cvss31_impactScore = cve.metrics.cvssMetricV31[0].impactScore
                    cvss31_exploitabilityScore = cve.metrics.cvssMetricV31[0].exploitabilityScore
                    cvss31_attackVector = cve.metrics.cvssMetricV31[0].cvssData.attackVector
                    cvss31_attackComplexity = cve.metrics.cvssMetricV31[0].cvssData.attackComplexity
                    cvss31_privilegesRequired = cve.metrics.cvssMetricV31[0].cvssData.privilegesRequired
                    cvss31_userInteraction = cve.metrics.cvssMetricV31[0].cvssData.userInteraction
                    cvss31_scope = cve.metrics.cvssMetricV31[0].cvssData.scope
                    cvss31_confidentialityImpact = cve.metrics.cvssMetricV31[0].cvssData.confidentialityImpact
                    cvss31_integrityImpact = cve.metrics.cvssMetricV31[0].cvssData.integrityImpact
                    cvss31_availabilityImpact = cve.metrics.cvssMetricV31[0].cvssData.availabilityImpact
                    cvss31_source = cve.metrics.cvssMetricV31[0].source
                # Mapeamos variables CNA
                if len(cve.metrics.cvssMetricV31)==2 and cve.metrics.cvssMetricV31[1].source == 'cna@vuldb.com':
                    cna_baseScore =  cve.metrics.cvssMetricV31[1].cvssData.baseScore
                    cna_baseSeverity = cve.metrics.cvssMetricV31[1].cvssData.baseSeverity
                    cna_vectorString = cve.metrics.cvssMetricV31[1].cvssData.vectorString
                    cna_version = cve.metrics.cvssMetricV31[1].cvssData.version
                    cna_impactScore = cve.metrics.cvssMetricV31[1].impactScore
                    cna_exploitabilityScore = cve.metrics.cvssMetricV31[1].exploitabilityScore
                    cna_attackVector = cve.metrics.cvssMetricV31[1].cvssData.attackVector
                    cna_attackComplexity = cve.metrics.cvssMetricV31[1].cvssData.attackComplexity
                    cna_privilegesRequired = cve.metrics.cvssMetricV31[1].cvssData.privilegesRequired
                    cna_userInteraction = cve.metrics.cvssMetricV31[1].cvssData.userInteraction
                    cna_scope = cve.metrics.cvssMetricV31[1].cvssData.scope
                    cna_confidentialityImpact = cve.metrics.cvssMetricV31[1].cvssData.confidentialityImpact
                    cna_integrityImpact = cve.metrics.cvssMetricV31[1].cvssData.integrityImpact
                    cna_availabilityImpact = cve.metrics.cvssMetricV31[1].cvssData.availabilityImpact
                    cna_source = cve.metrics.cvssMetricV31[1].source

            if hasattr(cve.metrics, 'cvssMetricV2'):
                    cvss20_baseScore =  cve.metrics.cvssMetricV2[0].cvssData.baseScore
                    cvss20_baseSeverity = cve.metrics.cvssMetricV2[0].baseSeverity
                    cvss20_vectorString = cve.metrics.cvssMetricV2[0].cvssData.vectorString
                    cvss20_impactSubscore = cve.metrics.cvssMetricV2[0].impactScore
                    cvss20_exploitabilitySubscore = cve.metrics.cvssMetricV2[0].exploitabilityScore
                    cvss20_accessVector = cve.metrics.cvssMetricV2[0].cvssData.accessVector
                    cvss20_accessComplexity = cve.metrics.cvssMetricV2[0].cvssData.accessComplexity
                    cvss20_authentication = cve.metrics.cvssMetricV2[0].cvssData.authentication
                    cvss20_confidentialityImpact = cve.metrics.cvssMetricV2[0].cvssData.confidentialityImpact
                    cvss20_integrityImpact = cve.metrics.cvssMetricV2[0].cvssData.integrityImpact
                    cvss20_availabilityImpact = cve.metrics.cvssMetricV2[0].cvssData.availabilityImpact

    header = f"""---
CP Source: "NIST"
CP Execution date: {'"'+str(datetime.datetime.now().strftime('%Y-%m-%d'))+'"'}
Vulnerability ID: {'"'+str(cve.id)+'"'}
Date: {'"'+str(datetime.datetime.strptime(cve.published, '%Y-%m-%dT%H:%M:%S.%f').strftime('%Y-%m-%d'))+'"'}
CVSS 3.x Base Score: {'"'+str(cvss31_baseScore)} {str(cvss31_baseSeverity)+'"'}
CVSS 3.x Vector: {'"'+str(cvss31_vectorString)+'"'}
CVSS 3.x Version: {'"'+str(cvss31_version)+'"'}
CVSS 3.x Impact Score: {'"'+str(cvss31_exploitabilityScore)+'"'}
CVSS 3.x Exploitability Score: {'"'+str(cvss31_exploitabilityScore)+'"'}
CVSS 3.x Attack Vector (AV): {'"'+str(cvss31_attackVector)+'"'}
CVSS 3.x Attack Complexity (AC): {'"'+str(cvss31_attackComplexity)+'"'}
CVSS 3.x Privileges Required (PR): {'"'+str(cvss31_privilegesRequired)+'"'}
CVSS 3.x User Interaction (UI): {'"'+str(cvss31_userInteraction)+'"'}
CVSS 3.x Scope (S): {'"'+str(cvss31_scope)+'"'}
CVSS 3.x Confidentiality (C): {'"'+str(cvss31_confidentialityImpact)+'"'}
CVSS 3.x Integrity (I): {'"'+str(cvss31_integrityImpact)+'"'}
CVSS 3.x Availability (A): {'"'+str(cvss31_availabilityImpact)+'"'}
CVSS 3.x Source: {'"'+str(cvss31_source)+'"'}

CNA Base Score: {'"'+str(cna_baseSeverity)} {str(cna_baseScore)+'"'}
CNA Vector: {'"'+str(cna_vectorString)+'"'}
CNA Version: {'"'+str(cna_version)+'"'}
CNA Impact Score: {'"'+str(cna_exploitabilityScore)+'"'}
CNA Exploitability Score: {'"'+str(cna_exploitabilityScore)+'"'}
CNA Attack Vector (AV): {'"'+str(cna_attackVector)+'"'}
CNA Attack Complexity (AC): {'"'+str(cna_attackComplexity)+'"'}
CNA Privileges Required (PR): {'"'+str(cna_privilegesRequired)+'"'}
CNA User Interaction (UI): {'"'+str(cna_userInteraction)+'"'}
CNA Scope (S): {'"'+str(cna_scope)+'"'}
CNA Confidentiality (C): {'"'+str(cna_confidentialityImpact)+'"'}
CNA Integrity (I): {'"'+str(cna_integrityImpact)+'"'}
CNA Availability (A): {'"'+str(cna_availabilityImpact)+'"'}
CNA Source: {'"'+str(cna_source)+'"'}

CVSS 2.x Base Score: {'"'+str(cvss20_baseScore)} {str(cvss20_baseSeverity)+'"'}
CVSS 2.x Vector: {'"'+str(cvss20_vectorString)+'"'}
CVSS 2.x Impact Subscore: {'"'+str(cvss20_impactSubscore)+'"'}
CVSS 2.x Exploitability Subscore: {'"'+str(cvss20_exploitabilitySubscore)+'"'}
CVSS 2.x Access Vector (AV): {'"'+str(cvss20_accessVector)+'"'}
CVSS 2.x Access Complexity (AC): {'"'+str(cvss20_accessComplexity)+'"'}
CVSS 2.x Authentication (AU): {'"'+str(cvss20_authentication)+'"'}
CVSS 2.x Confidentiality (C): {'"'+str(cvss20_confidentialityImpact)+'"'}
CVSS 2.x Integrity (I): {'"'+str(cvss20_integrityImpact)+'"'}
CVSS 2.x Availability (A): {'"'+str(cvss20_availabilityImpact)+'"'}
---
"""
    return header

In [15]:
def write_dict_to_md(cve, write_name):
    '''
    Esta es la función principal de construcción de las vulnerabilidades. Es llamada dentro de un bucle que recorre el conjunto de vulnerabilidades obtenidas por get_nist_vulnerabilities() y se encarga de crear el archivo en la ruta especificada, llamar al constructor de las propiedades create_header_properties(), añadir los elementos restantes definidos al archivo y guardar y cerrar la escritura del mismo.
    '''
    try:
        with open(write_name, 'w', encoding='utf-8') as file:
            v_dict = cve.__dict__
            header_properties = create_header_properties(cve)
            file.write(header_properties+ "\n")
            file.write('**'+'Vulnerability ID' + '**' + "\n")
            file.write(cve.id+ "\n")
            file.write("\n"+'**'+'Description' + '**' + "\n")
            file.write(cve.descriptions[0].value + "\n") # Ojo, chequear items sin descripción
            file.write("\n"+'**'+'References to Advisories, Solutions, and Tools'+'**'+ "\n")
            if len(cve.references)>0:
                for r in range(len(cve.references)):
                    file.write(cve.references[r].source +' - '+cve.references[r].url+ "\n")
            file.write("\n"+ '**' + 'Weakness Enumeration'+'**'+ "\n")
            if hasattr(cve, 'weaknesses'):
                if len(cve.weaknesses) > 0:
                    for w in range(len(cve.weaknesses)):
                        file.write(cve.weaknesses[0].description[0].value +' - '+ 'https://cwe.mitre.org/data/definitions/'+cve.weaknesses[0].description[0].value[4:]+'.html')
        print(f"Archivo .md guardado en: {write_name}")
    except (OSError, IOError) as e:
        print(f"Error al escribir en el archivo {write_name}: {e}")

#### **Ejecución principal**

In [16]:
try:
    vulnerabilities = get_nist_vulnerabilities(start_date, end_date)
    
except requests.exceptions.ReadTimeout:
    print("Error: La solicitud ha fallado después de varios intentos debido a un tiempo de espera.")
else:
    print(f"Se han obtenido {str(len(vulnerabilities))} vulnerabilidades entre las fechas {start_date.strftime('%d/%m/%Y')} y {end_date.strftime('%d/%m/%Y')}")

Se han obtenido 121 vulnerabilidades entre las fechas 01/07/2024 y 02/07/2024


#### **Guardado**

In [17]:
# Recorremos el listado de vulnerabilidades
for v in vulnerabilities:
    # Chequeamos que la fecha de publicación para el guardado y asignamos nombre del fichero final
    if v.published != '':
        folder_name = os.path.join(os.getcwd(), 'outputs', 'nist', 'save_by_date', datetime.datetime.strptime(v.published, '%Y-%m-%dT%H:%M:%S.%f').strftime('%Y%m%d'))
        # file_name = v.id +'_'+ datetime.datetime.strptime(v.published, '%Y-%m-%dT%H:%M:%S.%f').strftime('%Y%m%d_%H%M')+'h.md' #temporal
        file_name = v.id + '_' + datetime.datetime.strptime(v.published, '%Y-%m-%dT%H:%M:%S.%f').strftime('%Y%m%d_%H%Mh') +'.md' 
    else:
        folder_name = os.path.join(os.getcwd(), 'outputs', 'nist', 'save_by_date', 'no-pubish-date')
        file_name =   v.id + '_' + 'no-pubish-date'+'.md'
    # Creamos carpeta si no existe
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    # Controlamos el guardado de los archivos mediante un parámetro
    if save_as_md:
        write_name = os.path.join(folder_name, file_name)
        write_dict_to_md(v, write_name)

Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-0153_20240701_0915h.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-39427_20240701_0915h.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-39428_20240701_0915h.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-39429_20240701_0915h.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-39430_20240701_0915h.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\NIST\save_by_date\20240701\CVE-2024-38987_20240701_1315h.md
Archi